# Moneyball Scout Agent — 37% Rule to Agentic AI

This notebook trains a simple predictive model, scores 100 sequential prospects, applies the classic secretary-problem stopping rule, and writes an agent-ready decision table to a Fabric Lakehouse.

**Decision:** observe, select, or keep searching.

**Important:** the 37% rule is a benchmark under strict secretary-problem assumptions, not a universal business rule.

In [ ]:
import math, mlflow, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

# OPTION A (recommended in Fabric): upload the two CSVs to the Lakehouse Files area
historical = pd.read_csv('/lakehouse/default/Files/moneyball_historical_players.csv')
incoming = pd.read_csv('/lakehouse/default/Files/moneyball_incoming_prospects.csv')
historical.head()

In [ ]:
features = ['age','obp','slg','walk_pct','strikeout_pct','exit_velocity','defensive_runs','projected_salary_m']
X = historical[features]
y = historical['high_value']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=37, stratify=y)

model = Pipeline([
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=37))
])
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:,1]
pred = (proba >= 0.5).astype(int)
print('ROC-AUC:', round(roc_auc_score(y_test, proba), 3))
print('Accuracy:', round(accuracy_score(y_test, pred), 3))

In [ ]:
# Track and register in Fabric MLflow
mlflow.set_experiment('Moneyball_Decision_Science')
with mlflow.start_run(run_name='logistic_regression_baseline'):
    mlflow.log_param('decision_rule', 'secretary_37_percent')
    mlflow.log_metric('roc_auc', float(roc_auc_score(y_test, proba)))
    mlflow.log_metric('accuracy', float(accuracy_score(y_test, pred)))
    mlflow.sklearn.log_model(model, artifact_path='model', registered_model_name='MoneyballProspectValueModel')

In [ ]:
# Score the incoming sequential candidate stream
incoming['predicted_high_value_probability'] = model.predict_proba(incoming[features])[:,1]
incoming['predicted_value_per_million'] = incoming['predicted_high_value_probability'] / incoming['projected_salary_m']

n = len(incoming)
observe_n = math.floor(n / math.e)  # ~= 36.8%
threshold = incoming.iloc[:observe_n]['predicted_high_value_probability'].max()

incoming['phase'] = np.where(incoming['arrival_order'] <= observe_n, 'OBSERVE', 'DECIDE')
incoming['reference_threshold'] = threshold
incoming['beats_reference'] = incoming['predicted_high_value_probability'] > threshold

eligible = incoming[(incoming['arrival_order'] > observe_n) & incoming['beats_reference']]
selected_order = int(eligible.iloc[0]['arrival_order']) if len(eligible) else int(incoming.iloc[-1]['arrival_order'])
incoming['decision'] = 'WAIT'
incoming.loc[incoming['arrival_order'] <= observe_n, 'decision'] = 'OBSERVE'
incoming.loc[incoming['arrival_order'] == selected_order, 'decision'] = 'SELECT'
incoming.loc[incoming['arrival_order'] > selected_order, 'decision'] = 'STOPPED'

selected = incoming[incoming['decision']=='SELECT'].iloc[0]
print(f'Observe first {observe_n} of {n} prospects.')
print(f'Reference threshold: {threshold:.3f}')
print(f"Select {selected['player_id']} at arrival {selected_order} with predicted probability {selected['predicted_high_value_probability']:.3f}.")

In [ ]:
# Optional prediction-aware decision guardrails
# Require the prospect to beat the observation threshold AND clear a minimum confidence/value test.
min_confidence = 0.65
min_value_per_million = 0.18
incoming['agent_recommendation'] = np.where(
    (incoming['arrival_order'] > observe_n) &
    (incoming['predicted_high_value_probability'] > threshold) &
    (incoming['predicted_high_value_probability'] >= min_confidence) &
    (incoming['predicted_value_per_million'] >= min_value_per_million),
    'SELECT',
    np.where(incoming['arrival_order'] <= observe_n, 'LEARN', 'KEEP_SEARCHING')
)
incoming[['player_id','arrival_order','predicted_high_value_probability','projected_salary_m','predicted_value_per_million','phase','decision','agent_recommendation']].head(45)

In [ ]:
# Write a Delta table for Power BI / Fabric Data Agent
spark_df = spark.createDataFrame(incoming)
spark_df.write.mode('overwrite').format('delta').saveAsTable('moneyball_decision_candidates')

# A compact summary table makes agent grounding even easier.
summary = pd.DataFrame([{
    'candidate_count': n,
    'observe_count': observe_n,
    'observe_fraction': observe_n/n,
    'reference_threshold': float(threshold),
    'selected_player_id': selected['player_id'],
    'selected_arrival_order': selected_order,
    'selected_probability': float(selected['predicted_high_value_probability'])
}])
spark.createDataFrame(summary).write.mode('overwrite').format('delta').saveAsTable('moneyball_decision_summary')

## Fabric Data Agent instructions

Create a Fabric Data Agent and add the Lakehouse tables `moneyball_decision_candidates` and `moneyball_decision_summary`.

Use instructions similar to:

> You are a decision-science analyst for a baseball scouting simulation. Explain recommendations using the candidate's predicted probability, arrival order, 37% observation threshold, salary, and value-per-million. Treat the 37% rule as a benchmark under secretary-problem assumptions, not a universal law. Never invent missing statistics. When asked whether to select a candidate, answer SELECT, KEEP SEARCHING, or LEARN first, followed by a short rationale.

Suggested test question:

> We have one roster spot and cannot recall rejected prospects. Based on the current candidate stream, should we select now or keep searching? Explain the threshold and prediction behind the recommendation.